# 1.4 — Hypothesis Testing & Confidence Intervals (drill version)

Same topic as before, broken into smaller segments with repeated practice at each one. Every concept below gets: a short explanation, one worked example showing the syntax, and **2-3 reps** — the same skill applied to a different feature or a different nuance (small sample, one-tailed, ties, sparse categories...). The point isn't to get through the notebook once; it's to do each rep cold, then check it, until the pattern is automatic.

Structure per rep: **Predict** (commit to an expectation) → **Task** (scaffolded, you fill in the `...`) → **Decision** (one or two sentences, same discipline as your EDA notebook).

**Data:** loads `application_train.csv` via relative path if present, otherwise falls back to a synthetic dataset with the same columns so the notebook still runs end to end.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')


# Project Setup

## Project Details (Add Project Name)


**Dataset:** German Credit Dataset — UCI 1994  
**Author:** Chuch  
**Date:** 2026-06-08  
**Objective:** What question is this notebook answering?  
**Status:** In Progress / Complete

## Environment and Imports

In [ ]:
# ══════════════════════════════════════════════════════
# Environment Setup
# ══════════════════════════════════════════════════════

# ── Standard Library ──────────────────────────────────
import os
import sys
import warnings

# ── Data ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ── Machine Learning ──────────────────────────────────
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Jupyter ───────────────────────────────────────────
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display

def find_repo_root(start, repo_name='CreditRiskLearning'):
    path = os.path.abspath(start)
    while True:
        if os.path.basename(path) == repo_name:
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root: {repo_name}")
        path = parent

repo_root = find_repo_root(os.getcwd())
sys.path.append(os.path.join(repo_root, 'Core Resources'))
sys.path.append(os.path.join(repo_root, 'Core Resources', 'Scripts'))

import python_style_util as psu
import evaluation_utils as eu
import missingness_viz as mv        # missingess assessment functions

# ══════════════════════════════════════════════════════
# Display Settings
# ══════════════════════════════════════════════════════

%matplotlib inline

# Pandas
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.colheader_justify', 'left')



# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Style Guide

In [ ]:
psu.show_tokens()
psu.show_fonts()


In [ ]:
import os

core = os.path.join(os.path.abspath('..'), 'Core Resources')
scripts = os.path.join(os.path.abspath('..'), 'Core Resources', 'Scripts')

print("Core Resources contents:")
print(os.listdir(core))

print("\nScripts contents:")
print(os.listdir(scripts) if os.path.exists(scripts) else "Scripts folder doesn't exist")

## Display Setttings

In [ ]:
# --- Pandas ---

pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places on floats
pd.set_option('display.max_columns', None)              # show all columns
pd.set_option('display.max_rows', 100)                  # show up to 100 rows
pd.set_option('display.width', None)                    # don't wrap wide dataframes
pd.set_option('display.colheader_justify', 'left')      # left-align column headers
pd.set_option('display.precision', 4)                   # decimal precision for describe()
pd.set_option('display.large_repr', 'truncate')         # truncate instead of summary for large dfs

# --- NumPy ---

np.set_printoptions(
    precision=4,        # decimal places (you have this)
    suppress=True,      # no scientific notation (you have this)
    linewidth=120,      # wrap width (you have this)
    threshold=1000,     # show up to 1000 elements before summarising with ...
    edgeitems=5,        # show 5 items at each end when it does summarise
)

# --- Matplotlib ---

%matplotlib inline
plt.rcParams['figure.dpi'] = 120             # sharper figures
plt.rcParams['savefig.dpi'] = 150            # higher res when saving
plt.rcParams['savefig.bbox'] = 'tight'       # no clipped labels when saving
plt.rcParams['savefig.facecolor'] = 'white'  # white background on saved figures

# --- Jupyter ---

InteractiveShell.ast_node_interactivity = 'all'  # print every expression, not just the last

# --- Warnings ---

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# --- Style ---

psu.set_style()

## Data Loading

In [ ]:
# Build path relative to notebook location
base_path = os.path.dirname(os.path.abspath('this_notebook.ipynb'))
data_path = os.path.join(base_path, 'German Credit Data', 'german.data')

# Load data
df = pd.read_csv('../Home Credit Default Risk/application_train.csv', sep=',')

In [ ]:
def find_repo_root(marker="CreditRiskLearning"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if parent.name == marker:
            return parent
    return path

def generate_synthetic_home_credit(n=8000, seed=RANDOM_STATE):
    """Stand-in for application_train.csv -- same key columns, similar
    relationships to TARGET. Unused once you point data_path at the real file."""
    rng = np.random.default_rng(seed)

    ext_source_1 = np.clip(rng.normal(0.50, 0.18, n), 0.01, 0.99)
    ext_source_2 = np.clip(rng.normal(0.51, 0.16, n), 0.01, 0.99)
    ext_source_3 = np.clip(rng.normal(0.49, 0.19, n), 0.01, 0.99)

    age_years = rng.uniform(21, 69, n)
    days_birth = -(age_years * 365.25).astype(int)

    employed_years = np.clip(rng.exponential(6, n), 0, 45)
    days_employed = -(employed_years * 365.25).astype(int)
    anomaly_mask = rng.random(n) < 0.05
    days_employed[anomaly_mask] = 365243

    income = rng.lognormal(mean=11.9, sigma=0.45, size=n)
    credit = income * rng.uniform(2, 6, n)
    annuity = credit / rng.uniform(10, 30, n)

    gender = rng.choice(['F', 'M'], size=n, p=[0.66, 0.34])
    contract_type = rng.choice(['Cash loans', 'Revolving loans'], size=n, p=[0.90, 0.10])
    education = rng.choice(
        ['Secondary / secondary special', 'Higher education', 'Incomplete higher', 'Lower secondary'],
        size=n, p=[0.71, 0.24, 0.03, 0.02]
    )
    family_status = rng.choice(
        ['Married', 'Single / not married', 'Civil marriage', 'Separated', 'Widow'],
        size=n, p=[0.64, 0.15, 0.10, 0.07, 0.04]
    )
    own_car = rng.choice(['Y', 'N'], size=n, p=[0.34, 0.66])
    children = rng.poisson(0.4, n)

    z_ext = (0.45*ext_source_1 + 0.35*ext_source_2 + 0.20*ext_source_3 - 0.5) / 0.15
    z_income = (np.log(income) - np.log(income).mean()) / np.log(income).std()
    z_age = (age_years - age_years.mean()) / age_years.std()
    edu_effect = np.where(education == 'Higher education', -0.35,
                  np.where(education == 'Lower secondary', 0.25, 0.0))
    contract_effect = np.where(contract_type == 'Revolving loans', -0.25, 0.0)
    employed_effect = np.where(days_employed == 365243, 0.15, -0.10*np.log1p(employed_years))

    logit = (-2.3 - 0.9*z_ext - 0.15*z_income - 0.25*z_age
             + edu_effect + contract_effect + employed_effect)
    prob_default = 1 / (1 + np.exp(-logit))
    target = rng.binomial(1, np.clip(prob_default, 0.01, 0.9))

    return pd.DataFrame({
        'SK_ID_CURR': np.arange(100001, 100001+n),
        'TARGET': target,
        'NAME_CONTRACT_TYPE': contract_type,
        'CODE_GENDER': gender,
        'FLAG_OWN_CAR': own_car,
        'CNT_CHILDREN': children,
        'AMT_INCOME_TOTAL': income.round(2),
        'AMT_CREDIT': credit.round(2),
        'AMT_ANNUITY': annuity.round(2),
        'NAME_EDUCATION_TYPE': education,
        'NAME_FAMILY_STATUS': family_status,
        'DAYS_BIRTH': days_birth,
        'DAYS_EMPLOYED': days_employed,
        'EXT_SOURCE_1': ext_source_1.round(4),
        'EXT_SOURCE_2': ext_source_2.round(4),
        'EXT_SOURCE_3': ext_source_3.round(4),
    })

repo_root = find_repo_root()
data_path = repo_root / "Data" / "application_train.csv"  # adjust if your folder name differs

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded real Home Credit data from {data_path}: {df.shape}")
else:
    print(f"{data_path} not found -- using synthetic stand-in so the notebook still runs.")
    df = generate_synthetic_home_credit()
    print(f"Synthetic data: {df.shape}")

df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25
df['YEARS_EMPLOYED'] = np.where(df['DAYS_EMPLOYED'] == 365243, np.nan, -df['DAYS_EMPLOYED'] / 365.25)

df[['TARGET', 'AMT_INCOME_TOTAL', 'EXT_SOURCE_2', 'AGE_YEARS']].describe()


## 1.4.1 — Framing H0 and H1

H0 is always the "no real effect" default; H1 is "the effect is real." Getting this backwards is the single most common hypothesis-testing mistake, so this section is just reps of writing them out, no code yet.


### Rep 1

Question: *does average income differ between defaulters and non-defaulters?*

Write H0 and H1 in your own words below.


*(H0: ... H1: ...)*

### Rep 2

Question: *is default rate associated with education level (categorical, not a mean comparison)?*

Write H0 and H1 for this one. Notice how the wording has to change now that neither variable is a continuous mean.


*(H0: ... H1: ...)*

### Rep 3 — spot the error

A colleague writes: *"H0: income and default are related. H1: income and default are not related."*

What's wrong with this framing? Rewrite it correctly.


*(What's wrong, and the corrected H0/H1.)*

## 1.4.2 — Type I vs Type II errors

Type I: reject H0 when it's actually true (false positive). Type II: fail to reject H0 when H1 is actually true (false negative/miss). Same two categories, different scenario each rep — the goal is recognising the pattern regardless of the story around it.


### Rep 1

You test a candidate scorecard feature against TARGET, get a significant result, add it to the model — but the feature isn't actually predictive in the population. Which error is this, and what's the downstream cost to the bank?


*(Answer.)*

### Rep 2

`DAYS_EMPLOYED` genuinely predicts default in the population, but your test on a small sample comes back non-significant, so you drop it from consideration. Which error, and what's the downstream cost this time?


*(Answer.)*

### Rep 3

A fraud-detection rule flags a legitimate transaction as fraudulent and blocks it. Which error type, in hypothesis-testing terms (treat H0 as "transaction is legitimate")? What lever (alpha, sample size, rule design) would you pull to reduce this specific error, and what does that do to the other error rate?


*(Answer.)*

## 1.4.3 — What a p-value means

p-value = probability of a difference at least this extreme, *if H0 were true*. Not the probability H0 is true. Not a measure of how big the effect is (that's Section 1.4.8). Building this from a permutation test first, before using `scipy` directly, makes the definition concrete rather than memorised.


In [ ]:
# Worked example: permutation test for income vs TARGET
observed_diff = df.loc[df['TARGET']==0, 'AMT_INCOME_TOTAL'].mean() - df.loc[df['TARGET']==1, 'AMT_INCOME_TOTAL'].mean()

rng = np.random.default_rng(RANDOM_STATE)
n_permutations = 2000
shuffled_diffs = np.empty(n_permutations)

income = df['AMT_INCOME_TOTAL'].values
target = df['TARGET'].values.copy()

for i in range(n_permutations):
    rng.shuffle(target)
    shuffled_diffs[i] = income[target==0].mean() - income[target==1].mean()

perm_p_value = (np.abs(shuffled_diffs) >= np.abs(observed_diff)).mean()
print(f"Observed difference: {observed_diff:,.0f}")
print(f"Permutation p-value: {perm_p_value:.4f}")

plt.figure(figsize=(7,4))
plt.hist(shuffled_diffs, bins=40, color='steelblue', alpha=0.7)
plt.axvline(observed_diff, color='crimson', linewidth=2, label='Observed')
plt.axvline(-observed_diff, color='crimson', linewidth=2, linestyle='--')
plt.legend(); plt.title('Null distribution from permutation'); plt.xlabel('Mean income difference')
plt.show()


### Rep 1 — same test, different feature

Predict first: do you expect `EXT_SOURCE_2`'s permutation p-value to be smaller or larger than income's above? (It's a manufactured risk score, so think about how directly it should relate to TARGET.) Then run it.


In [ ]:
# Rep 1: permutation test for EXT_SOURCE_2 vs TARGET
observed_diff_ext = ...  # TODO
shuffled_diffs_ext = ...  # TODO: reuse the loop pattern above
perm_p_value_ext = ...  # TODO

print(f"Permutation p-value: {perm_p_value_ext}")


*(Decision + was your prediction right?)*

### Rep 2 — same test, smaller sample

Predict first: if you rerun the income permutation test on a random subsample of just 300 rows (same underlying effect, less data), does the p-value go up, down, or stay about the same? Then test it.


In [ ]:
# Rep 2: permutation test on a 300-row subsample of income vs TARGET
df_small = df.sample(n=300, random_state=RANDOM_STATE)

observed_diff_small = ...  # TODO
shuffled_diffs_small = ...  # TODO
perm_p_value_small = ...  # TODO

print(f"Permutation p-value (n=300): {perm_p_value_small}")
print(f"Permutation p-value (full sample): {perm_p_value}")


*(Decision: what does this rep tell you about the relationship between sample size and p-value, holding the true effect constant?)*

## 1.4.4 — Two-sample t-test

`scipy.stats.ttest_ind(a, b, equal_var=False)` runs Welch's t-test (unequal variances assumed, the safer default). Three reps below each change one thing about the setup so you can feel what actually shifts the result.


In [ ]:
# Worked example: income by TARGET, Welch's t-test, two-tailed
income_0 = df.loc[df['TARGET']==0, 'AMT_INCOME_TOTAL'].dropna()
income_1 = df.loc[df['TARGET']==1, 'AMT_INCOME_TOTAL'].dropna()

t_stat, p_value = stats.ttest_ind(income_0, income_1, equal_var=False)
print(f"Welch's t-test -- t: {t_stat:.3f}, p: {p_value:.4f}")


### Rep 1 — Welch's vs Student's

Run the same income comparison with `equal_var=True` (Student's t-test) instead. Predict first: given the group sizes are very unequal (far fewer defaulters than non-defaulters), do you expect the p-value to move much?


In [ ]:
# Rep 1: Student's t-test (equal_var=True) on the same income comparison
t_stat_student, p_value_student = ...  # TODO

print(f"Student's t-test -- t: {t_stat_student}, p: {p_value_student}")
print(f"Welch's t-test (from above) -- p: {p_value}")


*(Decision: how much did the p-value move, and does that match what you'd expect from unequal group sizes/variances?)*

### Rep 2 — one-tailed vs two-tailed

`YEARS_EMPLOYED` vs TARGET. If your hypothesis is specifically *"defaulters have been employed for fewer years"* (directional), you'd want a one-tailed test. `scipy.stats.ttest_ind` is two-tailed by default -- for one-tailed, halve the two-tailed p-value if the difference is in the predicted direction (or use `alternative='less'`/`'greater'`).

Run both and compare.


In [ ]:
# Rep 2: two-tailed vs one-tailed t-test for YEARS_EMPLOYED vs TARGET
employed_0 = df.loc[df['TARGET']==0, 'YEARS_EMPLOYED'].dropna()
employed_1 = df.loc[df['TARGET']==1, 'YEARS_EMPLOYED'].dropna()

t_stat_two, p_value_two = ...  # TODO: two-tailed (default)
t_stat_one, p_value_one = ...  # TODO: one-tailed, alternative='greater' or 'less' -- pick the direction matching "defaulters employed fewer years"

print(f"Two-tailed p: {p_value_two}")
print(f"One-tailed p: {p_value_one}")


*(Decision: which direction did you pick for the one-tailed test, and why? How does the one-tailed p-value relate numerically to the two-tailed one?)*

### Rep 3 — small sample

Repeat the income vs TARGET t-test on a 40-row random subsample (`df.sample(n=40, random_state=RANDOM_STATE)`). Predict first: with a much smaller n, do you expect the p-value to be more or less stable if you re-ran this with a different random seed?


In [ ]:
# Rep 3: t-test on a 40-row subsample
df_tiny = df.sample(n=40, random_state=RANDOM_STATE)
income_0_tiny = df_tiny.loc[df_tiny['TARGET']==0, 'AMT_INCOME_TOTAL'].dropna()
income_1_tiny = df_tiny.loc[df_tiny['TARGET']==1, 'AMT_INCOME_TOTAL'].dropna()

t_stat_tiny, p_value_tiny = ...  # TODO

print(f"n=40 p-value: {p_value_tiny}")
print(f"group sizes: {len(income_0_tiny)} vs {len(income_1_tiny)}")


*(Decision: try re-running with a different random_state -- how much does the p-value swing? What does that tell you about trusting a single small-sample test?)*

## 1.4.5 — Confidence intervals

A p-value answers "is there a difference?" A confidence interval answers "how big is it, plausibly?" -- and it's the more useful number for a business decision. A 95% CI for a mean difference that excludes 0 corresponds to a significant result at alpha=0.05 -- same information, more useful shape.

Two ways to build one: the analytic formula (t-distribution) and bootstrap resampling (fewer assumptions, more compute). Worth doing both once so you see they agree.


In [ ]:
# Worked example: 95% CI for the income difference (TARGET=0 minus TARGET=1), analytic + bootstrap
# Analytic (Welch-Satterthwaite via statsmodels-free approximation using scipy)
mean_diff = income_0.mean() - income_1.mean()
se_diff = np.sqrt(income_0.var(ddof=1)/len(income_0) + income_1.var(ddof=1)/len(income_1))
dof = (income_0.var(ddof=1)/len(income_0) + income_1.var(ddof=1)/len(income_1))**2 / (
    (income_0.var(ddof=1)/len(income_0))**2/(len(income_0)-1) + (income_1.var(ddof=1)/len(income_1))**2/(len(income_1)-1)
)
t_crit = stats.t.ppf(0.975, dof)
ci_analytic = (mean_diff - t_crit*se_diff, mean_diff + t_crit*se_diff)
print(f"Analytic 95% CI for mean difference: ({ci_analytic[0]:,.0f}, {ci_analytic[1]:,.0f})")

# Bootstrap
rng = np.random.default_rng(RANDOM_STATE)
n_boot = 2000
boot_diffs = np.empty(n_boot)
for i in range(n_boot):
    boot_0 = rng.choice(income_0, size=len(income_0), replace=True)
    boot_1 = rng.choice(income_1, size=len(income_1), replace=True)
    boot_diffs[i] = boot_0.mean() - boot_1.mean()
ci_boot = (np.percentile(boot_diffs, 2.5), np.percentile(boot_diffs, 97.5))
print(f"Bootstrap 95% CI for mean difference: ({ci_boot[0]:,.0f}, {ci_boot[1]:,.0f})")
print(f"Does the CI exclude 0? {'yes -- consistent with p < 0.05' if ci_analytic[0]*ci_analytic[1] > 0 else 'no -- consistent with p > 0.05'}")


### Rep 1 — CI for a different feature

Build the bootstrap 95% CI for the `EXT_SOURCE_2` mean difference between TARGET groups. Check whether it excludes 0 and whether that matches the t-test conclusion from Section 1.4.3/1.4.4-style test on this feature.


In [ ]:
# Rep 1: bootstrap CI for EXT_SOURCE_2 mean difference
ext2_0 = df.loc[df['TARGET']==0, 'EXT_SOURCE_2'].dropna()
ext2_1 = df.loc[df['TARGET']==1, 'EXT_SOURCE_2'].dropna()

boot_diffs_ext2 = ...  # TODO: reuse the bootstrap loop pattern
ci_boot_ext2 = ...  # TODO

print(f"Bootstrap 95% CI: {ci_boot_ext2}")


*(Decision: does the CI exclude 0? Does that match a t-test on the same data?)*

### Rep 2 — CI for a proportion

Default rate is a proportion, not a mean -- different CI method. `statsmodels.stats.proportion.proportion_confint(count, nobs, method='wilson')` gives a Wilson score interval, which behaves better than the normal approximation for proportions near 0 or 1 (like an ~8% default rate).

Build a 95% CI for the default rate among borrowers with `NAME_EDUCATION_TYPE == 'Higher education'`, and a separate one for everyone else. Do the two intervals overlap?


In [ ]:
# Rep 2: Wilson CI for default rate by education group
higher_ed = df[df['NAME_EDUCATION_TYPE'] == 'Higher education']
other_ed = df[df['NAME_EDUCATION_TYPE'] != 'Higher education']

count_higher = ...  # TODO: number of defaults in higher_ed
nobs_higher = ...  # TODO: number of rows in higher_ed
ci_higher = ...  # TODO: proportion_confint(count_higher, nobs_higher, method='wilson')

count_other = ...  # TODO
nobs_other = ...  # TODO
ci_other = ...  # TODO

print(f"Higher education default rate 95% CI: {ci_higher}")
print(f"Other education default rate 95% CI: {ci_other}")


*(Decision: do the intervals overlap? What would you conclude about education as a risk driver from this alone, versus from the chi-square test in 1.4.6?)*

## 1.4.6 — Chi-square test of independence

`pd.crosstab` builds the contingency table, `scipy.stats.chi2_contingency` runs the test. The assumption underneath: expected cell counts should generally be >=5. When they're not, the chi-square approximation gets unreliable and `scipy.stats.fisher_exact` (2x2 tables only) is the safer choice.


In [ ]:
# Worked example: NAME_CONTRACT_TYPE vs TARGET
contingency = pd.crosstab(df['NAME_CONTRACT_TYPE'], df['TARGET'])
print(contingency)

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nchi2 = {chi2_stat:.3f}, p = {p_value:.4f}, dof = {dof}")
print(f"Smallest expected count: {expected.min():.1f}")


### Rep 1 — different categorical pair

Run the same test for `NAME_FAMILY_STATUS` vs `TARGET` (more than 2 categories this time -- check the contingency table shape and degrees of freedom change accordingly).


In [ ]:
# Rep 1: chi-square for NAME_FAMILY_STATUS vs TARGET
contingency_fam = ...  # TODO
chi2_stat_fam, p_value_fam, dof_fam, expected_fam = ...  # TODO

print(f"chi2 = {chi2_stat_fam}, p = {p_value_fam}, dof = {dof_fam}")


*(Decision.)*

### Rep 2 — small expected counts

Take a 60-row random subsample of the data (`df.sample(n=60, random_state=RANDOM_STATE)`) and rebuild the `NAME_CONTRACT_TYPE` vs `TARGET` contingency table. Check the expected counts -- if any are below 5, run `scipy.stats.fisher_exact` on the 2x2 table instead of chi-square, and compare the two p-values.


In [ ]:
# Rep 2: small-sample contingency table -- chi-square vs Fisher's exact
df_small_cat = df.sample(n=60, random_state=RANDOM_STATE)
contingency_small = ...  # TODO

chi2_stat_small, p_value_small, dof_small, expected_small = ...  # TODO
print(f"Smallest expected count: {expected_small.min()}")

odds_ratio, p_value_fisher = ...  # TODO: stats.fisher_exact(contingency_small)
print(f"Chi-square p: {p_value_small}")
print(f"Fisher's exact p: {p_value_fisher}")


*(Decision: were the expected counts too small for chi-square here? Did the two p-values agree?)*

## 1.4.7 — Mann-Whitney U (non-parametric alternative)

Ranks instead of means, no normality assumption -- the tool to reach for when a distribution is heavily skewed or you're not confident the CLT has kicked in.


In [ ]:
# Worked example: AMT_INCOME_TOTAL, checked visually, then Mann-Whitney vs t-test
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].hist(df['AMT_INCOME_TOTAL'], bins=60, color='steelblue')
axes[0].set_title('AMT_INCOME_TOTAL distribution')
stats.probplot(df['AMT_INCOME_TOTAL'], dist='norm', plot=axes[1])
plt.tight_layout(); plt.show()

u_stat, p_value_mw = stats.mannwhitneyu(income_0, income_1, alternative='two-sided')
print(f"Mann-Whitney p: {p_value_mw:.4f}  |  t-test p (from 1.4.4): {p_value:.4f}")


### Rep 1 — different skewed feature

Check `AMT_CREDIT`'s shape, then run both Mann-Whitney and a t-test comparing it across TARGET groups. Do the two agree on significance?


In [ ]:
# Rep 1: AMT_CREDIT -- distribution check + both tests
# TODO: histogram + Q-Q plot for AMT_CREDIT

credit_0 = ...  # TODO
credit_1 = ...  # TODO

u_stat_credit, p_value_mw_credit = ...  # TODO
t_stat_credit, p_value_t_credit = ...  # TODO

print(f"Mann-Whitney p: {p_value_mw_credit}, t-test p: {p_value_t_credit}")


*(Decision.)*

### Rep 2 — a feature with heavy ties

`CNT_CHILDREN` takes only a handful of integer values (0, 1, 2, 3...) so many rows share the exact same value -- lots of ties in the ranking Mann-Whitney relies on. Run it anyway and compare against a t-test.


In [ ]:
# Rep 2: CNT_CHILDREN -- ties in the ranks
children_0 = ...  # TODO
children_1 = ...  # TODO

u_stat_children, p_value_mw_children = ...  # TODO
t_stat_children, p_value_t_children = ...  # TODO

print(f"Mann-Whitney p: {p_value_mw_children}, t-test p: {p_value_t_children}")
print(f"Unique values in CNT_CHILDREN: {sorted(df['CNT_CHILDREN'].unique())}")


*(Decision: does having lots of tied values change how much you'd trust the Mann-Whitney result here?)*

## 1.4.8 — Effect size vs statistical significance

Cohen's d (mean comparisons) and Cramer's V (categorical association) tell you *how big*, independent of *how certain*. The clearest way to internalise the gap between the two: hold the effect size fixed and watch the p-value move as N changes.


In [ ]:
# Worked example: Cohen's d for income by TARGET
def cohens_d(a, b):
    n_a, n_b = len(a), len(b)
    pooled_std = np.sqrt(((n_a-1)*a.std(ddof=1)**2 + (n_b-1)*b.std(ddof=1)**2) / (n_a+n_b-2))
    return (a.mean() - b.mean()) / pooled_std

d = cohens_d(income_0, income_1)
print(f"Cohen's d (full sample, n={len(income_0)+len(income_1)}): {d:.3f}")
print(f"t-test p-value (full sample): {p_value:.4f}")


### Rep 1 — same effect size, shrink N

Recompute Cohen's d and the t-test p-value for income vs TARGET on the 300-row subsample (`df_small`) from Section 1.4.3, Rep 2. Compare the effect size and p-value against the full-sample numbers above.


In [ ]:
# Rep 1: Cohen's d + p-value on the 300-row subsample
income_0_small = df_small.loc[df_small['TARGET']==0, 'AMT_INCOME_TOTAL'].dropna()
income_1_small = df_small.loc[df_small['TARGET']==1, 'AMT_INCOME_TOTAL'].dropna()

d_small = ...  # TODO
t_stat_small, p_value_small_d = ...  # TODO

print(f"Cohen's d (n=300): {d_small}")
print(f"p-value (n=300): {p_value_small_d}")


*(Decision: did the effect size stay roughly the same while the p-value moved? What's the practical lesson for reading p-values from very large or very small samples?)*

### Rep 2 — Cramer's V for a categorical pair

Compute Cramer's V for `NAME_CONTRACT_TYPE` vs `TARGET` (from 1.4.6). Formula: `V = sqrt(chi2 / (n * min(rows-1, cols-1)))`.


In [ ]:
# Rep 2: Cramer's V for NAME_CONTRACT_TYPE vs TARGET
n_total = contingency.values.sum()
r, k = contingency.shape
cramers_v = ...  # TODO

print(f"Cramer's V: {cramers_v}")


*(Decision: statistically significant, practically meaningful, both, or neither?)*

## 1.4.9 — Multiple comparisons correction

Test enough features and some will look significant by noise alone. Bonferroni (family-wise error rate, conservative) and Benjamini-Hochberg (false discovery rate, less conservative) both correct for this -- but how much they bite depends on how many tests you're running.


In [ ]:
# Worked example: t-test a batch of numeric features against TARGET, Bonferroni-correct
numeric_features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'CNT_CHILDREN']
raw_p_values = []
for feature in numeric_features:
    g0 = df.loc[df['TARGET']==0, feature].dropna()
    g1 = df.loc[df['TARGET']==1, feature].dropna()
    _, p = stats.ttest_ind(g0, g1, equal_var=False)
    raw_p_values.append(p)

reject_bonf, p_bonf, _, _ = multipletests(raw_p_values, alpha=0.05, method='bonferroni')
pd.DataFrame({'feature': numeric_features, 'raw_p': raw_p_values, 'bonferroni_p': p_bonf, 'significant': reject_bonf})


### Rep 1 — same batch, Benjamini-Hochberg instead

Apply `method='fdr_bh'` to the same `raw_p_values` list above. Which features (if any) flip from non-significant to significant compared to Bonferroni?


In [ ]:
# Rep 1: Benjamini-Hochberg on the same p-values
reject_bh, p_bh, _, _ = ...  # TODO

pd.DataFrame({'feature': numeric_features, 'raw_p': raw_p_values, 'bonferroni_significant': reject_bonf, 'bh_significant': reject_bh})


*(Decision.)*

### Rep 2 — family size changes the correction's bite

Bonferroni-correct just the first 3 features in `numeric_features`, then separately correct all 7. Compare the corrected p-value for `AMT_INCOME_TOTAL` between the two runs -- same raw p-value, different correction.


In [ ]:
# Rep 2: same feature, correction strength depends on family size
raw_p_first_3 = raw_p_values[:3]
raw_p_all_7 = raw_p_values

_, p_bonf_3, _, _ = ...  # TODO: bonferroni on raw_p_first_3
_, p_bonf_7, _, _ = ...  # TODO: bonferroni on raw_p_all_7

print(f"AMT_INCOME_TOTAL Bonferroni p (family of 3): {p_bonf_3[0]}")
print(f"AMT_INCOME_TOTAL Bonferroni p (family of 7): {p_bonf_7[0]}")


*(Decision: why did the same raw p-value produce a different corrected p-value? What does this imply about how you scope a feature screen before running it?)*

## 1.4.10 — Capstone reps

Three short capstones instead of one long one. For each: state H0/H1, pick the right test (justify by data type and shape), run it, add an effect size or CI, and write a decision.


### Capstone rep 1 — pick a numeric feature not yet tested (e.g. `AMT_ANNUITY`, `EXT_SOURCE_1`)

In [ ]:
# Capstone rep 1
# TODO


*(Decision.)*

### Capstone rep 2 — pick a categorical feature not yet tested (e.g. `CODE_GENDER`, `FLAG_OWN_CAR`)

In [ ]:
# Capstone rep 2
# TODO


*(Decision.)*

### Capstone rep 3 — your choice, any feature, but justify your test choice explicitly before running anything

In [ ]:
# Capstone rep 3
# TODO


*(Decision.)*